# LaDe-P Demand Forecasting — XGBoost Model

## Goal
Train and evaluate an XGBoost regression model to predict `demand_count` for each `(city, region_id, aoi_id, bucket_hour)` using lag and rolling features.

## Assumptions
- `model_df` is already loaded (from the preprocessing notebook output `lade_hourly_features.csv`).
- All lag/rolling features are pre-computed and NaN rows have been dropped.
- `bucket_hour` is a datetime column.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Install xgboost if needed (Colab usually has it, but just in case)
try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "xgboost", "-q"])
    import xgboost as xgb
    print("XGBoost installed, version:", xgb.__version__)

SEED = 42
np.random.seed(SEED)
print("Ready.")

## Step 1 — Load the Preprocessed Dataset

If `model_df` is already in memory (you ran the preprocessing notebook in the same session), skip the CSV read.
Otherwise it reads `lade_hourly_features.csv` from disk.

In [ ]:
CSV_PATH = "lade_hourly_features.csv"

if 'model_df' not in globals():
    print("Loading from CSV...")
    model_df = pd.read_csv(CSV_PATH, parse_dates=["bucket_hour"])
    print("Loaded shape:", model_df.shape)
else:
    print("Using existing model_df from memory, shape:", model_df.shape)
    model_df["bucket_hour"] = pd.to_datetime(model_df["bucket_hour"])

model_df.head()

## Step 2 — Chronological Train / Val / Test Split

We split **globally** by time (not per AOI), so:
- **Train**: first 60% of hours
- **Val**: next 20% of hours
- **Test**: last 20% of hours

This prevents any future data from leaking into training.

In [ ]:
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20

all_hours = model_df["bucket_hour"].sort_values().unique()
n_hours   = len(all_hours)

train_end = all_hours[int(n_hours * TRAIN_RATIO) - 1]
val_end   = all_hours[int(n_hours * (TRAIN_RATIO + VAL_RATIO)) - 1]

def assign_split(bh):
    if bh <= train_end:
        return "train"
    elif bh <= val_end:
        return "val"
    else:
        return "test"

model_df["split"] = model_df["bucket_hour"].map(assign_split)

split_counts = model_df["split"].value_counts().sort_index()
print(split_counts)
print(f"\nTrain ends:  {train_end}")
print(f"Val ends:    {val_end}")
print(f"Test starts: {model_df.loc[model_df['split']=='test','bucket_hour'].min()}")

## Step 3 — Feature Engineering

- **Drop** `bucket_hour` as a direct feature (time info is captured by `hour`, `dow`, `month`).
- **One-hot encode** `city` and `aoi_type` (high-cardinality `aoi_id` and `region_id` are kept as numeric IDs).

In [ ]:
TARGET = "demand_count"

# Columns to drop (not used as features)
DROP_COLS = ["bucket_hour", "split", TARGET]

# One-hot encode low-cardinality categoricals
OHE_COLS = ["city", "aoi_type"]

encoded = pd.get_dummies(model_df.drop(columns=DROP_COLS), columns=OHE_COLS, drop_first=True)

FEATURE_COLS = encoded.columns.tolist()
print(f"Number of features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

## Step 4 — Prepare Train / Val / Test Arrays

- **Train**: sample up to 2M rows (to keep RAM manageable).
- **Val / Test**: use full split if memory allows, otherwise sample similarly.

In [ ]:
MAX_TRAIN_ROWS = 2_000_000

split_mask = model_df["split"]

train_idx = model_df.index[split_mask == "train"]
val_idx   = model_df.index[split_mask == "val"]
test_idx  = model_df.index[split_mask == "test"]

# Sample train rows if too large
if len(train_idx) > MAX_TRAIN_ROWS:
    train_idx = train_idx[np.random.RandomState(SEED).choice(
        len(train_idx), MAX_TRAIN_ROWS, replace=False
    )]
    print(f"Train sampled to {MAX_TRAIN_ROWS:,} rows")
else:
    print(f"Train rows: {len(train_idx):,}")

X_train = encoded.loc[train_idx]
y_train = model_df.loc[train_idx, TARGET]

X_val   = encoded.loc[val_idx]
y_val   = model_df.loc[val_idx, TARGET]

X_test  = encoded.loc[test_idx]
y_test  = model_df.loc[test_idx, TARGET]

print(f"Val rows:  {len(X_val):,}")
print(f"Test rows: {len(X_test):,}")

## Step 5 — Baseline Models

Two simple baselines:
- **Baseline A** (`lag_24`): predict using the value from 24 hours ago.
- **Baseline B** (`roll_24_mean`): predict using the 24-hour rolling average.

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    # Avoid division by zero (when both actual and forecast are zero)
    mask = denom > 0
    return np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def evaluate(y_true, y_pred, name, split_name):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return {
        "Model": name,
        "Split": split_name,
        "MAE":   round(mae(y_true, y_pred), 4),
        "RMSE":  round(rmse(y_true, y_pred), 4),
        "sMAPE": round(smape(y_true, y_pred), 2),
    }

results = []

for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test)]:
    results.append(evaluate(y_s, X_s["lag_24"],       "Baseline lag_24",      split_name))
    results.append(evaluate(y_s, X_s["roll_24_mean"], "Baseline roll_24_mean", split_name))

print(pd.DataFrame(results))

## Step 6 — Train XGBoost with Early Stopping

In [ ]:
# Use gpu_hist if a GPU is available, otherwise fall back to hist
try:
    import subprocess
    gpu_check = subprocess.run(["nvidia-smi"], capture_output=True)
    tree_method = "gpu_hist" if gpu_check.returncode == 0 else "hist"
except Exception:
    tree_method = "hist"

print(f"Using tree_method='{tree_method}'")

model = xgb.XGBRegressor(
    objective       = "reg:squarederror",
    tree_method     = tree_method,
    n_estimators    = 1000,      # upper bound; early stopping will cut this
    learning_rate   = 0.05,
    max_depth       = 6,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    min_child_weight= 5,
    reg_alpha       = 0.1,
    reg_lambda      = 1.0,
    random_state    = SEED,
    n_jobs          = -1,
    early_stopping_rounds = 30,
    eval_metric     = "rmse",
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f"\nBest iteration: {model.best_iteration}")

## Step 7 — Evaluate XGBoost on Val and Test

In [ ]:
val_preds  = model.predict(X_val)
test_preds = model.predict(X_test)

# Clip negatives — demand cannot be < 0
val_preds  = np.clip(val_preds,  0, None)
test_preds = np.clip(test_preds, 0, None)

results.append(evaluate(y_val,  val_preds,  "XGBoost", "val"))
results.append(evaluate(y_test, test_preds, "XGBoost", "test"))

# Pivot into a clean comparison table
results_df = pd.DataFrame(results)
comparison = results_df.pivot(index="Model", columns="Split", values=["MAE", "RMSE", "sMAPE"])
comparison.columns = [f"{metric}_{split}" for metric, split in comparison.columns]
comparison = comparison[["MAE_val", "RMSE_val", "sMAPE_val", "MAE_test", "RMSE_test", "sMAPE_test"]]

print("\n=== Model Comparison ===")
print(comparison.to_string())

## Step 8 — Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("XGBoost — Top 20 Feature Importances")
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.show()

## Step 9 — Example AOI Plot: Actual vs Predicted

Pick the busiest AOI in the test set and plot actual demand vs XGBoost prediction vs lag_24 baseline.

In [ ]:
# Attach predictions back to the test portion of model_df
test_slice = model_df.loc[test_idx].copy()
test_slice["pred_xgb"]   = test_preds
test_slice["pred_lag24"] = X_test["lag_24"].values

# Choose the AOI with the highest total demand in test
best_aoi = (
    test_slice.groupby(["city", "region_id", "aoi_id"])["demand_count"]
    .sum()
    .idxmax()
)
city_s, region_s, aoi_s = best_aoi

aoi_test = (
    test_slice[
        (test_slice["city"] == city_s) &
        (test_slice["region_id"] == region_s) &
        (test_slice["aoi_id"] == aoi_s)
    ]
    .sort_values("bucket_hour")
)

# Limit plot to first 14 days for readability
plot_days = 14
aoi_test_plot = aoi_test.head(24 * plot_days)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["demand_count"],
        label="Actual",   color="black",      linewidth=1.5)
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["pred_xgb"],
        label="XGBoost",  color="steelblue",  linewidth=1.2, linestyle="--")
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["pred_lag24"],
        label="Lag-24",   color="tomato",     linewidth=1.0, linestyle=":")

ax.set_title(f"Hourly Demand Forecast — AOI {aoi_s} (region {region_s}, {city_s})\nTest set, first {plot_days} days")
ax.set_xlabel("Time")
ax.set_ylabel("demand_count")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\nSelected AOI: city={city_s}, region_id={region_s}, aoi_id={aoi_s}")
print(f"Test rows for this AOI: {len(aoi_test)}")